# 01 · The modelling table

What the models learn from: one row per loan, with the member's record **before**
the loan was given and how the loan ended. Built by dbt
(`analytics.loan_features`) and exported with `python -m chama.pipeline export`.

> The table holds member codes and features only: no names, no free text (decision D-016). The default label is **provisional** until the Chama agrees a definition (D-018): `defaulted` = rolled over while unpaid, overdue, or lost track.

In [ ]:
# Setup: runs in Google Colab and on this computer.
# Colab, once: add a GitHub token that can read this repository as the Colab
# secret GITHUB_TOKEN (key icon on the left). This cell mounts Drive, fetches
# the code and installs what Colab lacks. On this computer it uses DRIVE_DIR
# from .env and the `chama` conda environment.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "m2-features-models"

if IN_COLAB:
    from google.colab import drive, userdata

    REPO = "/content/chama-loan-default"
    OWNER = "muriithi-alvin"
    PLAIN_URL = f"https://github.com/{OWNER}/chama-loan-default.git"

    def git(*args):
        """Run git with the token; show git's message, never the token."""
        token = userdata.get("GITHUB_TOKEN").strip()
        url = PLAIN_URL.replace("https://", f"https://{OWNER}:{token}@")
        result = subprocess.run(["git", *[url if a == "URL" else a for a in args]],
                                capture_output=True, text=True)
        if result.returncode:
            message = (result.stderr or result.stdout).replace(token, "***").strip()
            raise RuntimeError(f"git failed: {message}")

    drive.mount("/content/drive")
    if os.path.exists(REPO):
        git("-C", REPO, "pull", "-q", "URL", BRANCH)
    else:
        git("clone", "-q", "--branch", BRANCH, "URL", REPO)
        git("-C", REPO, "remote", "set-url", "origin", PLAIN_URL)   # no token left behind
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlflow", "shap", "imbalanced-learn", "python-dotenv"], check=True)
    sys.path.insert(0, f"{REPO}/src")
    os.environ["DRIVE_DIR"] = "/content/drive/MyDrive/chama-loan-default"

import warnings

import pandas as pd

from chama.config import export_dir, load_config, mlflow_dir
from chama.features.export import FILE_NAME

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
CONFIG = load_config()
SETTINGS = CONFIG["modelling"]
DATA = export_dir() / FILE_NAME
print("data:  ", DATA)
print("MLflow:", mlflow_dir())

In [ ]:
from chama.models.data import FEATURES, TARGET, known_outcomes, load_features, load_manifest

df = load_features(DATA)
manifest = load_manifest(DATA)
{k: manifest.get(k) for k in ["exported_at", "rows", "training_rows", "defaults",
                              "members", "default_statuses", "git_commit"]}

## Loans and defaults per year

Only loans with a known outcome (`in_training`) are used for learning. The walk-forward
test scores each year from 2019 to 2024 with a model trained on earlier years.

In [ ]:
known = known_outcomes(df)
by_year = known.groupby("issue_year").agg(loans=(TARGET, "size"), defaults=(TARGET, "sum"))
by_year["default_rate"] = (by_year["defaults"] / by_year["loans"]).round(2)
by_year["tested"] = by_year.index.isin(SETTINGS["test_years"])
by_year

In [ ]:
by_type = known.groupby("loan_type").agg(loans=(TARGET, "size"), defaults=(TARGET, "sum"))
by_type["default_rate"] = (by_type["defaults"] / by_type["loans"]).round(2)
by_type

## Missing values

Empty is meaningful here: `amount_vs_own_max` is empty for a member's first loan of a
type, `recent_contribution_rate` before contributions were written down (2017). The
models fill gaps with the training median and add a column saying the value was missing.

In [ ]:
known[FEATURES].isna().mean().sort_values(ascending=False).round(2).to_frame("share_missing")

## Features by outcome

A first look at which features differ between repaid and defaulted loans. Differences
here are hints, not proof: the models and SHAP (notebook 03) weigh them together.

In [ ]:
numeric = [c for c in FEATURES if c != "loan_type"]
comparison = known.groupby(TARGET)[numeric].mean().T.round(2)
comparison.columns = ["repaid (0)", "defaulted (1)"]
comparison